# fingerprinting_02. Optimal fingerprinting: figures

Everything here is drawn from what notebook fingerprinting_01 saved, so nothing slow runs. fingerprinting_01 explains each step, with a quick look at each; the sections here keep its numbers. Each figure's size comes from `charts.chart_figure_size` and its panels from a GridSpec (`fig.add_gridspec(...).subplots()`, in a figure made with `layout='constrained'`); each panel is drawn by `extant/plots/fingerprinting.py` (`fpp`).

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import xarray as xr

from extant import config
from extant.logger import setup_logging
from extant.plots import charts, fingerprinting as fpp, forced_response as fr

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

In [ ]:
#(t): As in fingerprinting_01: the variable, the final fit, the signal set shown step by step
VARIABLE = config.VARIABLES['tas']
METHOD = 'tls'
MAIN_SET = '4 signals'

#(t): The colour scale of the tables of trends
trend_scale = fr.Scale(np.round(np.arange(-0.6, 0.61, 0.1), 2), 'RdBu_r', f'Trend ({VARIABLE.units} per decade)')

In [ ]:
#(t): Where fingerprinting_01 saved its results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

## Which Results

Run **one** of the next two sections: each sets the folder the results are opened from.

### Sample data

In [ ]:
#(t): The sample's results
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
RESULTS_DIR

### Full data

In [ ]:
#(t): The full data's results
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
RESULTS_DIR

## Open the Results

First what fingerprinting_01 saved in `RESULTS_DIR`, then each result opened.

In [ ]:
#(t): What fingerprinting_01 saved in RESULTS_DIR
RESULTS_DIR, sorted(path.name for path in RESULTS_DIR.glob('fingerprinting_*.zarr'))

In [ ]:
#(t): β and its 90% range, on (set, fit, signal)
betas_ds = xr.open_zarr(RESULTS_DIR / 'fingerprinting_betas.zarr').load()

In [ ]:
#(t): The attributable trends of MAIN_SET, on (signal, region, season)
attributable_ds = xr.open_zarr(RESULTS_DIR / 'fingerprinting_attributable.zarr').load()

In [ ]:
#(t): The tables of regions x seasons: ERA5's trends, MAIN_SET's fingerprints, its fit and residual
tables_ds = xr.open_zarr(RESULTS_DIR / 'fingerprinting_tables.zarr').load()

In [ ]:
#(t): The residual test of every set
residual_tests_ds = xr.open_zarr(RESULTS_DIR / 'fingerprinting_residual_tests.zarr').load()

In [ ]:
#(t): The collinearity of MAIN_SET
collinearity_ds = xr.open_zarr(RESULTS_DIR / 'fingerprinting_collinearity.zarr').load()

In [ ]:
#(t): β with each model's own fingerprints, and leaving each model out
robustness_ds = xr.open_zarr(RESULTS_DIR / 'fingerprinting_robustness.zarr').load()

## 4. Fingerprints

The observed trends and each fingerprint of `MAIN_SET`, as tables of regions × seasons on one colour scale.

In [ ]:
#(t): The observations and each fingerprint of MAIN_SET, as tables of regions x seasons, on one colour scale
observation_tables = {'ERA5': tables_ds['era5'],
                      **{str(s): tables_ds['fingerprint'].sel(signal=s) for s in tables_ds['signal'].values}}
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(observation_tables), n_legends=0,
                                                  n_colorbars=1, panel_width=1.9, panel_height=1.8, left_margin=1.6,
                                                  column_gap=0.2), layout='constrained')
axes = fig.add_gridspec(1, len(observation_tables)).subplots(sharey=True, squeeze=False)
for ax, (label, table_da) in zip(axes[0], observation_tables.items()):
    table_image = fpp.plot_entry_table(table_da, trend_scale, ax=ax)
    ax.set_title(label, loc='left')
fig.colorbar(table_image, ax=axes, location='bottom', shrink=0.6, label=trend_scale.label)
fig.suptitle(f'The observations and the fingerprints ({MAIN_SET})', x=0.01, ha='left');

## 8. Uncertainty

β and its 90% range for every signal set: clear of 0, detected; across 1, the models' response has the observed size.

In [ ]:
#(t): β and its 90% range for every signal set (rows), OLS (circles) and TLS (diamonds): clear of 0, detected; across 1
#(t): (dashed), the model's response has the observed size
signal_sets = list(betas_ds['set'].values)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(signal_sets), n_cols=1, n_legends=1, panel_width=5.5,
                                                  panel_height=1.3, left_margin=1.0), layout='constrained')
axes = fig.add_gridspec(len(signal_sets), 1).subplots(sharex=True, squeeze=False)
for ax, signal_set in zip(axes[:, 0], signal_sets):
    fpp.plot_scaling_factors(betas_ds.sel(set=signal_set), ax=ax)
    ax.set_title(str(signal_set), loc='left')
axes[-1, 0].set_xlabel('Scaling factor β (90% range)')
fig.legend(handles=[Line2D([], [], linestyle='none', marker=fpp.METHOD_MARKERS.get(str(method), 'o'),
                           color='0.3', label=str(method).upper()) for method in betas_ds['fit'].values],
           loc='outside lower center', ncols=betas_ds.sizes['fit'], frameon=False)
fig.suptitle('Scaling factors: every signal set, OLS and TLS (90% range)', x=0.01, ha='left');

## 9. Residual Consistency Test

The null distribution of each set's residual statistic, the observed one, and which entries drive the residual.

In [ ]:
#(t): The residual consistency test of each signal set: the null distribution (grey) and the observed statistic
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=residual_tests_ds.sizes['set'], n_legends=0, panel_width=2.6,
                                                  panel_height=2.0), layout='constrained')
axes = fig.add_gridspec(1, residual_tests_ds.sizes['set']).subplots(squeeze=False)
for ax, signal_set in zip(axes[0], residual_tests_ds['set'].values):
    set_test_ds = residual_tests_ds.sel(set=signal_set)
    fpp.plot_residual_null({'statistic': float(set_test_ds['statistic']), 'null': set_test_ds['null'].values}, ax=ax)
    ax.set_title(f"{signal_set}\np = {float(set_test_ds['pvalue']):.3f}", loc='left')
    ax.set_xlabel('rᵀ C₂⁻¹ r')
fig.suptitle(f'The residual consistency test ({METHOD.upper()})', x=0.01, ha='left');

In [ ]:
#(t): The observations, the fit and the residual, as tables of regions x seasons, on one colour scale
fit_tables = {'ERA5': tables_ds['era5'], 'fit': tables_ds['fit'], 'residual': tables_ds['residual']}
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(fit_tables), n_legends=0, n_colorbars=1,
                                                  panel_width=1.9, panel_height=1.8, left_margin=1.6, column_gap=0.2),
                 layout='constrained')
axes = fig.add_gridspec(1, len(fit_tables)).subplots(sharey=True, squeeze=False)
for ax, (label, table_da) in zip(axes[0], fit_tables.items()):
    table_image = fpp.plot_entry_table(table_da, trend_scale, ax=ax)
    ax.set_title(label, loc='left')
fig.colorbar(table_image, ax=axes, location='bottom', shrink=0.6, label=trend_scale.label)
fig.suptitle(f'The observations, the fit and the residual ({MAIN_SET}, {METHOD.upper()})', x=0.01, ha='left');

## 10. What It Says

Each signal's attributable trend, β times its fingerprint with the range of β, beside the observed trend.

In [ ]:
#(t): Each signal's attributable trend with its 90% range (bars), beside the observed (black diamonds): one row per
#(t): region, the seasons across
observed_trend_da = tables_ds['era5']
trend_regions = list(observed_trend_da['region'].values)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(trend_regions), n_cols=1, n_legends=1, panel_width=6.0,
                                                  panel_height=1.5, left_margin=1.6), layout='constrained')
axes = fig.add_gridspec(len(trend_regions), 1).subplots(sharex=True, squeeze=False)
for ax, region in zip(axes[:, 0], trend_regions):
    fpp.plot_attributable_trends(observed_trend_da.sel(region=region), attributable_ds.sel(region=region), ax=ax)
    ax.set_ylabel(f'{region}\n{VARIABLE.units} per decade')
trend_handles, trend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(trend_handles, trend_labels, loc='outside lower center', ncols=len(trend_labels), frameon=False)
fig.suptitle(f'Attributable trends ({MAIN_SET}, {METHOD.upper()}) and the observed (ERA5)', x=0.01, ha='left');

## 11. Collinearity

Above about 0.8 two signals cannot be told apart.

In [ ]:
#(t): Collinearity: the correlation of the fingerprints over the entries, and of the β's across the Monte Carlo trials;
#(t): above about 0.8 two signals cannot be told apart
correlation_matrices = {'Fingerprints (over the entries)': collinearity_ds['fingerprints'],
                        f'β ({METHOD.upper()}, across the trials)': collinearity_ds['beta']}
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(correlation_matrices), n_legends=0,
                                                  n_colorbars=1, panel_width=3.0, panel_height=3.0, column_gap=0.9),
                 layout='constrained')
axes = fig.add_gridspec(1, len(correlation_matrices)).subplots(squeeze=False)
for ax, (label, matrix_da) in zip(axes[0], correlation_matrices.items()):
    correlation_image = fpp.plot_correlations(matrix_da, ax=ax)
    ax.set_title(label, loc='left')
fig.colorbar(correlation_image, ax=axes, location='bottom', shrink=0.6, label='Correlation')
fig.suptitle(f'Collinearity ({MAIN_SET})', x=0.01, ha='left');

## 12. Robustness

β of the multi-model fingerprints, with each model's own fingerprints, and leaving each model out.

In [ ]:
#(t): MAIN_SET's β and its range by the final fit, and the two robustness checks, on (model, signal)
main_beta_ds = betas_ds.sel(set=MAIN_SET, fit=METHOD).dropna('signal')
each_model_beta_da = robustness_ds['each_model']
left_out_beta_da = robustness_ds['left_out']
main_beta_ds

In [ ]:
#(t): β of the multi-model fingerprints (bar: the 90% range), with each model's own fingerprints (markers), and with
#(t): each model left out (crosses)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=1, panel_width=5.5,
                                                  panel_height=0.6 * main_beta_ds.sizes['signal'] + 0.4),
                 layout='constrained')
axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
fpp.plot_model_betas(main_beta_ds, each_model_beta_da, left_out_beta_da, ax=axes[0, 0])
axes[0, 0].set_xlabel('Scaling factor β')
fig.legend(handles=charts.model_legend_handles(each_model_beta_da['model'].values)
           + [Line2D([], [], linestyle='none', marker='x', color='0.25', label='leaving that model out')],
           loc='outside lower center', ncols=each_model_beta_da.sizes['model'] + 1, frameon=False)
fig.suptitle(f'β with each model, and without it ({MAIN_SET}, {METHOD.upper()})', x=0.01, ha='left');